# CNN Training Analysis

**FILE:** `notebooks/cnn_training_analysis.ipynb`
**AUTHOR:** Kananelo Chabeli — University of Cape Town

Reads the training ledger and answers three questions:

1. **Is the experiment valid?** Did every model train under one recipe, and did any fail?
2. **Did the shared recipe suit every architecture?** A single recipe is what makes this an
   architectural comparison, but it will fit some lineages better than others. Where it does
   not, that is a finding to report, not a per-model hyperparameter to tune.
3. **What is the accuracy/cost frontier?** Which architectures are Pareto-optimal on parameters
   and operations before any hardware is involved.

Inputs: `results/training/<task>/summary.csv` and `epochs.csv`.

In [ ]:
import json, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

warnings.filterwarnings("ignore", category=FutureWarning)

# ----- configuration -------------------------------------------------------------------
TASK     = "classification"
ROOT     = Path.cwd().parent                 # notebooks/ sits one level below the repo root
RESULTS  = ROOT / "results" / "training" / TASK

SUMMARY_CSV = RESULTS / "summary.csv"
EPOCHS_CSV  = RESULTS / "epochs.csv"

FIGDIR = ROOT / "results" / "figures" / TASK / "training"
FIGDIR.mkdir(parents=True, exist_ok=True)
SAVE_FIGURES = True
DPI = 150

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": DPI, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "legend.frameon": False,
})

def save(fig, name):
    """Write a figure to the task's figure directory.

    Args:
        fig: Matplotlib figure.
        name: File stem, without extension.
    """
    if SAVE_FIGURES:
        fig.savefig(FIGDIR / f"{name}.png")
        fig.savefig(FIGDIR / f"{name}.pdf")   # vector copy for the write-up

def have(df, *cols):
    """Report whether a frame carries every named column.

    Args:
        df: DataFrame to check.
        *cols: Required column names.

    Returns:
        True when all are present.
    """
    return all(c in df.columns for c in cols)

print(f"summary: {SUMMARY_CSV}  exists={SUMMARY_CSV.exists()}")
print(f"epochs : {EPOCHS_CSV}   exists={EPOCHS_CSV.exists()}")

## 1. Load and validate

In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
epochs  = pd.read_csv(EPOCHS_CSV) if EPOCHS_CSV.exists() else pd.DataFrame()

ok   = summary[summary.status == "completed"].copy() if have(summary, "status") else summary.copy()
bad  = summary[summary.status != "completed"] if have(summary, "status") else summary.iloc[0:0]

print(f"{len(summary)} rows | completed {len(ok)} | failed {len(bad)}")
if len(bad):
    print("\nFAILED:")
    print(bad[["model", "error"]].to_string(index=False))

# A stable colour per model, reused in every figure here and in cnn_results.
MODELS  = sorted(ok.model.unique())
PALETTE = {m: cm.tab20(i % 20) for i, m in enumerate(MODELS)}
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
(ROOT / "results" / "figures" / "model_palette.json").write_text(
    json.dumps({m: list(c) for m, c in PALETTE.items()}, indent=2))
print(f"\n{len(MODELS)} models: {MODELS}")

### 1.1 Experiment validity

The single-recipe claim is the premise of every comparison below, so it is checked rather than
assumed. More than one recipe digest means the table mixes experimental conditions and its rows
are not comparable.

In [ ]:
checks = []

if have(ok, "recipe_digest"):
    digests = sorted(ok.recipe_digest.astype(str).unique())
    checks.append(("single recipe digest", len(digests) == 1, digests))

for col, label in [("epochs", "identical epoch count"),
                   ("seed", "identical seed"),
                   ("train_res", "identical train resolution"),
                   ("eval_res", "identical eval resolution"),
                   ("batch_size", "identical effective batch")]:
    if have(ok, col):
        vals = sorted(ok[col].dropna().unique().tolist())
        checks.append((label, len(vals) == 1, vals))

# Normalisation statistics are computed per micro-batch, so a differing micro-batch size is a
# small but real confound between models. Flagged rather than treated as fatal.
if have(ok, "micro_batch_size"):
    vals = sorted(ok.micro_batch_size.dropna().unique().tolist())
    checks.append(("identical micro-batch (BN statistics)", len(vals) == 1, vals))

print(f"{'check':42s} {'':4s} values")
print("-" * 78)
for label, passed, vals in checks:
    print(f"{label:42s} {'OK' if passed else 'WARN':4s} {vals}")

## 2. Learning curves — did the recipe suit every architecture?

In [ ]:
if epochs.empty:
    print("no per-epoch log; skipping curve analysis")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

    for m in MODELS:
        d = epochs[epochs.model == m].sort_values("epoch")
        if d.empty:
            continue
        c = PALETTE[m]
        axes[0].plot(d.epoch, d.top1, color=c, lw=1.3, label=m)
        axes[1].plot(d.epoch, d.train_loss, color=c, lw=1.3)
        # The gap between validation and training loss is the clearest single indicator of
        # whether the shared regularisation is too weak or too strong for an architecture.
        axes[2].plot(d.epoch, d.val_loss - d.train_loss, color=c, lw=1.3)

    axes[0].set(xlabel="epoch", ylabel="validation top-1 (%)", title="Validation accuracy")
    axes[1].set(xlabel="epoch", ylabel="training loss", title="Training loss")
    axes[2].set(xlabel="epoch", ylabel="val loss − train loss",
                title="Generalisation gap")
    axes[2].axhline(0, color="k", lw=0.8, ls="--")
    axes[0].legend(fontsize=6.5, ncol=2, loc="lower right")
    fig.suptitle("Training dynamics under one shared recipe", y=1.02)
    fig.tight_layout(); save(fig, "01_learning_curves"); plt.show()

### 2.1 Convergence behaviour

`epochs_to_90pct` is how long a model took to reach 90% of its own final accuracy — a measure of
convergence speed that is independent of how accurate it ended up.

`final_minus_best` is negative when the last epoch was worse than the best one. A large negative
value means the model was still unstable at the end of the schedule, which is a sign the shared
recipe does not suit it.

In [ ]:
if not epochs.empty:
    rows = []
    for m in MODELS:
        d = epochs[epochs.model == m].sort_values("epoch")
        if d.empty:
            continue
        final, best = d.top1.iloc[-1], d.top1.max()
        target = 0.9 * final
        reached = d[d.top1 >= target]
        rows.append({
            "model": m,
            "final_top1": round(final, 2),
            "best_top1": round(best, 2),
            "final_minus_best": round(final - best, 2),
            "epochs_to_90pct": int(reached.epoch.iloc[0]) if len(reached) else None,
            # Standard deviation over the last ten epochs: a plateau should be quiet.
            "last10_std": round(d.top1.tail(10).std(), 3),
            "final_gap": round(d.val_loss.iloc[-1] - d.train_loss.iloc[-1], 3),
        })
    conv = pd.DataFrame(rows).sort_values("final_top1", ascending=False)
    display(conv)

    unstable = conv[conv.final_minus_best < -1.0]
    if len(unstable):
        print("\nStill unstable at the end of the schedule (final more than 1 pp below best):")
        print(unstable[["model", "final_minus_best", "last10_std"]].to_string(index=False))

## 3. Final accuracy

In [ ]:
acc = ok.sort_values("top1", ascending=False)
fig, ax = plt.subplots(figsize=(9, max(3.5, 0.38 * len(acc))))
y = np.arange(len(acc))
ax.barh(y, acc.top1, color=[PALETTE[m] for m in acc.model], height=0.62, label="top-1")
if have(acc, "top5"):
    ax.barh(y, acc.top5, color="none", edgecolor="0.4", lw=0.8, height=0.62, label="top-5")
ax.set_yticks(y); ax.set_yticklabels(acc.model, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("accuracy (%)"); ax.set_title("Final-epoch validation accuracy")
ax.legend(loc="lower right")
for i, v in enumerate(acc.top1):
    ax.text(v + 0.4, i, f"{v:.1f}", va="center", fontsize=7.5)
fig.tight_layout(); save(fig, "02_accuracy_ranking"); plt.show()

cols = [c for c in ["model", "source", "top1", "top5", "best_top1", "best_epoch",
                    "train_minutes", "seconds_per_epoch"] if c in acc.columns]
display(acc[cols].reset_index(drop=True))

## 4. Accuracy against model cost

Parameters and operations are *theoretical* costs. Whether they predict real latency is the
question the hardware campaign answers; these plots establish the frontier before hardware is
involved, so any later reordering is attributable to the hardware.

In [ ]:
def pareto_front(df, x, y, x_lower_better=True):
    """Select the non-dominated points of a two-objective trade-off.

    Args:
        df: Frame containing the two columns.
        x: Cost column.
        y: Quality column, always higher-is-better.
        x_lower_better: Whether a smaller x is preferable.

    Returns:
        The non-dominated rows, sorted by x.
    """
    d = df.dropna(subset=[x, y]).sort_values(x, ascending=x_lower_better)
    keep, best = [], -np.inf
    for _, r in d.iterrows():
        if r[y] > best:
            keep.append(r)
            best = r[y]
    return pd.DataFrame(keep)


def tradeoff(ax, df, xcol, ycol, xlabel, ylabel, logx=False):
    """Draw a cost/quality scatter with its Pareto front.

    Args:
        ax: Target axes.
        df: Source frame.
        xcol: Cost column.
        ycol: Quality column.
        xlabel: X-axis label.
        ylabel: Y-axis label.
        logx: Use a logarithmic x-axis.
    """
    d = df.dropna(subset=[xcol, ycol])
    for _, r in d.iterrows():
        ax.scatter(r[xcol], r[ycol], s=68, color=PALETTE.get(r.model, "grey"),
                   edgecolor="k", lw=0.5, zorder=3)
        ax.annotate(r.model, (r[xcol], r[ycol]), fontsize=6.2,
                    xytext=(4, 4), textcoords="offset points")
    front = pareto_front(d, xcol, ycol)
    ax.plot(front[xcol], front[ycol], "k--", lw=1.1, alpha=0.65, zorder=2,
            label=f"Pareto front ({len(front)})")
    if logx:
        ax.set_xscale("log")
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.legend(loc="lower right")
    return front


# params_M and GMACs are produced by eval/baseline.py; merge them in when available.
BASE = ROOT / "results" / "benchmark" / TASK / "baseline_fp32.csv"
cost = ok.copy()
if BASE.exists():
    b = pd.read_csv(BASE)
    add = [c for c in ["params_M", "GMACs"] if c in b.columns]
    if add:
        cost = cost.merge(b[["model"] + add], on="model", how="left")
        print(f"merged {add} from the FP32 baseline")
else:
    print("baseline_fp32.csv not found — run eval/baseline.py for parameter and operation counts")

pairs = [(c, lab, log) for c, lab, log in
         [("params_M", "parameters (M)", False), ("GMACs", "multiply-accumulates (G)", True)]
         if c in cost.columns]

if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(7.2 * len(pairs), 5.2), squeeze=False)
    for ax, (c, lab, log) in zip(axes[0], pairs):
        front = tradeoff(ax, cost, c, "top1", lab, "top-1 (%)", logx=log)
        print(f"\nPareto-optimal on {lab}:")
        print(front[["model", c, "top1"]].to_string(index=False))
    fig.suptitle("Accuracy against theoretical cost", y=1.01)
    fig.tight_layout(); save(fig, "03_accuracy_vs_cost"); plt.show()

## 5. Training cost

In [ ]:
metrics = [(c, lab) for c, lab in [("seconds_per_epoch", "seconds per epoch"),
                                   ("train_minutes", "total minutes"),
                                   ("images_per_second", "training throughput (img/s)"),
                                   ("peak_memory_mb", "peak memory (MB)")]
           if c in ok.columns]

if metrics:
    n = len(metrics)
    fig, axes = plt.subplots(1, n, figsize=(4.6 * n, max(3.6, 0.3 * len(ok))), squeeze=False)
    for ax, (c, lab) in zip(axes[0], metrics):
        d = ok.dropna(subset=[c]).sort_values(c, ascending=False)
        ax.barh(np.arange(len(d)), d[c], color=[PALETTE[m] for m in d.model], height=0.65)
        ax.set_yticks(np.arange(len(d))); ax.set_yticklabels(d.model, fontsize=7)
        ax.invert_yaxis(); ax.set_xlabel(lab)
    fig.suptitle("Training cost on the development GPU", y=1.02)
    fig.tight_layout(); save(fig, "04_training_cost"); plt.show()

### 5.1 Does a theoretical cost predict training time?

A first, cheap test of the question the hardware campaign asks properly. Spearman is used rather
than Pearson because the relationship need not be linear and it is the *ordering* that matters.

In [ ]:
if have(cost, "GMACs", "seconds_per_epoch"):
    d = cost.dropna(subset=["GMACs", "seconds_per_epoch"])
    if len(d) >= 3:
        rho_mac = d.GMACs.corr(d.seconds_per_epoch, method="spearman")
        print(f"Spearman(GMACs, seconds_per_epoch)   = {rho_mac:+.3f}   n={len(d)}")
        if "params_M" in d.columns:
            print(f"Spearman(params, seconds_per_epoch)  = "
                  f"{d.params_M.corr(d.seconds_per_epoch, method='spearman'):+.3f}")
        fig, ax = plt.subplots(figsize=(6.2, 5))
        for _, r in d.iterrows():
            ax.scatter(r.GMACs, r.seconds_per_epoch, s=70, color=PALETTE.get(r.model, "grey"),
                       edgecolor="k", lw=0.5)
            ax.annotate(r.model, (r.GMACs, r.seconds_per_epoch), fontsize=6.2,
                        xytext=(4, 3), textcoords="offset points")
        ax.set(xscale="log", xlabel="multiply-accumulates (G)", ylabel="seconds per epoch",
               title=f"Theoretical cost vs training time (Spearman {rho_mac:+.2f})")
        fig.tight_layout(); save(fig, "05_macs_vs_traintime"); plt.show()
        print("\nA weak correlation here is the first evidence that operation counts do not "
              "predict real execution time, which the hardware campaign tests properly.")

## 6. Export the analysis table

In [ ]:
out = cost.copy()
if not epochs.empty:
    out = out.merge(conv.drop(columns=["final_top1", "best_top1"], errors="ignore"),
                    on="model", how="left")
keep = [c for c in ["model", "source", "provider_version", "recipe_digest", "top1", "top5",
                    "best_top1", "best_epoch", "final_minus_best", "epochs_to_90pct",
                    "last10_std", "final_gap", "params_M", "GMACs", "seconds_per_epoch",
                    "train_minutes", "peak_memory_mb"] if c in out.columns]
table = out[keep].sort_values("top1", ascending=False)
path = RESULTS / "training_analysis.csv"
table.to_csv(path, index=False)
print(f"wrote {path}")
display(table)

## 7. Notes for the write-up

- Accuracies are **final-epoch**, not best-epoch. Selecting the best epoch on the split being
  reported would be selecting on the test set; `best_top1` is retained only as a diagnostic.
- These are ImageNet-100 figures and are **not comparable to published ImageNet-1k numbers**.
  They are internally comparable, which is what the ranking question requires.
- Any architecture flagged as unstable in section 2.1 should be reported as such. A single
  shared recipe fitting some lineages better than others is a finding about applying one recipe
  across architectures, not a defect to be tuned away per model.
- cuDNN selects algorithms by timing, so two runs with one seed can differ slightly. Before
  treating a small accuracy gap between architectures as real, estimate that noise by repeating
  a few models under different seeds.